# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Install the mlcroissant library if not already available
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()

print(f"Dataset: {metadata_json['name']}")
print(f"Description: {metadata_json['description']}")
print(f"Published: {metadata_json.get('datePublished', 'N/A')}")
print(f"Keywords: {', '.join(metadata_json.get('keywords', []))}")

## 2. Data Overview
Explore available record sets and their schema. All entities are referenced by their `@id` fields, as defined in the Croissant metadata.

In [ ]:
# List available record sets. The Dataset object exposes them as .record_sets (list of RecordSet objects)
print("Available Record Sets (by @id):")
for rs in dataset.record_sets:
    print(f" - @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")

# Inspect fields for the main record set (by convention the main table is often under the first record set)
main_record_set = dataset.record_sets[0]  # Use the 1st as main for demonstration
main_record_set_id = main_record_set['@id']

print(f"\nFields for record set '@id': {main_record_set_id}")
for field in main_record_set['field']:
    field_obj = dataset._metadata.fields_by_id[field['@id']]
    print(f" • Field @id: {field_obj['@id']} | name: {field_obj.get('name', 'N/A')} | type: {field_obj.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data records for each record set into a DataFrame for analysis, referencing record set `@id` and field `@id` identifiers.

In [ ]:
# Gather all record set `@id`s
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# Load all tables into pandas DataFrames
dataframes = {}
for rs in dataset.record_sets:
    rs_id = rs['@id']
    print(f"Loading records for record set '{rs_id}'...")
    rows = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(rows)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")

# Example: display the first rows from the main record set
main_rs_id = record_set_ids[0]
print(f"\nFirst 5 rows from record set '{main_rs_id}':")
display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps: filtering, normalizing numeric fields, grouping, and basic summary statistics. All field references use their Croissant `@id`/field name.

In [ ]:
# For demonstration, choose a numeric field (e.g., 'phq9_score') and a categorical grouping field
df = dataframes[main_rs_id]

# Find a numeric field in the schema (by @id or name)
numeric_fields = []
group_fields = []
for field in dataset._metadata.fields_by_id.values():
    if field.get('dataType') in ['schema:Integer', 'schema:Float', 'Integer', 'Float']:
        numeric_fields.append(field['@id'])
    elif field.get('dataType') == 'schema:Text':
        group_fields.append(field['@id'])

# Example: Use PHQ-9 total score (commonly named 'phq9_score' or similar) and 'gender' as grouping if they exist
phq9_score_field = None
gender_field = None
for f in numeric_fields:
    if 'phq' in f.lower():
        phq9_score_field = f
        break
for f in group_fields:
    if 'gender' in f.lower():
        gender_field = f
        break

if not phq9_score_field:
    print("Could not locate PHQ-9 score field in the schema. Using first available numeric field.")
    phq9_score_field = numeric_fields[0] if numeric_fields else None
    print(f"Using field: {phq9_score_field}")
else:
    print(f"Using field for PHQ-9 score: {phq9_score_field}")

if not gender_field:
    print("Could not locate gender field in the schema. Using first available text field if present.")
    gender_field = group_fields[0] if group_fields else None
    print(f"Using field: {gender_field}")
else:
    print(f"Using field for gender: {gender_field}")

# Filter records with PHQ-9 score > 10 (a common threshold for moderate depression)
if phq9_score_field in df.columns:
    filtered_df = df[df[phq9_score_field] > 10].copy()
    print(f"Filtered {len(filtered_df)} records with {phq9_score_field} > 10.")

    # Normalize the numeric field (Z-score)
    norm_col = f"{phq9_score_field}_normalized"
    filtered_df[norm_col] = (filtered_df[phq9_score_field] - filtered_df[phq9_score_field].mean()) / filtered_df[phq9_score_field].std()
    print(f"First 5 normalized records:")
    display(filtered_df[[phq9_score_field, norm_col]].head())

    # If gender is available, group by it
    if gender_field and gender_field in filtered_df.columns:
        grouped = filtered_df.groupby(gender_field)[phq9_score_field].mean().reset_index()
        print(f"\nMean {phq9_score_field} grouped by {gender_field}:")
        display(grouped)
else:
    print(f"Field '{phq9_score_field}' not found in main DataFrame. Available columns: {df.columns.tolist()}")

## 5. Visualization
Visualize mental health score distributions and grouping by demographic variables (e.g., gender or age group).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of PHQ-9 scores
if phq9_score_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[phq9_score_field].dropna(), bins=12, kde=True, color="#009688")
    plt.title("Distribution of PHQ-9 Scores")
    plt.xlabel("PHQ-9 Score")
    plt.ylabel("Count")
    plt.show()

# Boxplot by Gender
if gender_field and phq9_score_field in df.columns and gender_field in df.columns:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=df[gender_field], y=df[phq9_score_field], palette="Set2")
    plt.title(f"{phq9_score_field} Distribution by {gender_field}")
    plt.xlabel(gender_field)
    plt.ylabel("PHQ-9 Score")
    plt.show()


## 6. Conclusion
This notebook demonstrated how to explore the FAIR² Kilifi Mental Health Survey dataset using the `mlcroissant` library. We loaded metadata, inspected schema via `@id` references, extracted records, and performed elementary EDA and visualization, all with reproducible, schema-driven steps.

**Next steps:** - Further analysis (e.g., on other scores like GAD-7/PSQ), advanced modeling, or integration with other datasets can be performed easily with this setup.